# 02 — PubMed Ingestion Pipeline

> **Stage:** Data Ingestion (RAG Knowledge Base)  
> **Source:** PubMed via BioPython (Entrez / Medline)

**Objective:** Build the ingestion pipeline to retrieve abstract-level medical literature. This data will eventually populate the Qdrant vector database for the RAG component.

Per the project's technical log (`DECISIONS.md`), we are adopting an **abstract-only retrieval strategy**. Abstracts are dense, structured, and highly available, fulfilling the clinical bibliographic assistance use case without the noise or access limitations of full-text articles.

---

**Sections**
1. Setup & Authentication
2. Exploring the Entrez API (MEDLINE format)
3. Building the Ingestion Function
4. Exporting Data

---

## 1. Setup & Authentication

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [9]:
from Bio import Entrez, Medline
from typing import List, Dict

Entrez.email = os.getenv("NCBI_EMAIL")
api_key = os.getenv("NCBI_API_KEY")
if api_key:
    Entrez.api_key = api_key

## 2. Exploring the Entrez API / MEDLINE Metadata
First do a manual test query to see what fields the MEDLINE format returns. This allows to map MEDLINE keys (like `TI`, `AB`, `MH`) to the target semantic schema.

In [21]:
# Fetching IDs for the query
query = "Colour perception deficiencies in children born prematurely"
max_results = 2

handle = Entrez.esearch(db="pubmed", term=query, retmax=max_results, sort="relevance")
record = Entrez.read(handle)
handle.close()

record

{'Count': '2', 'RetMax': '2', 'RetStart': '0', 'IdList': ['37775921', '23841870'], 'TranslationSet': [{'From': 'Colour perception', 'To': '"colour perception"[All Fields] OR "color perception"[MeSH Terms] OR ("color"[All Fields] AND "perception"[All Fields]) OR "color perception"[All Fields]'}, {'From': 'deficiencies', 'To': '"deficiences"[All Fields] OR "deficiencies"[All Fields] OR "deficiency"[Subheading] OR "deficiency"[All Fields] OR "deficient"[All Fields] OR "deficients"[All Fields]'}, {'From': 'children', 'To': '"child"[MeSH Terms] OR "child"[All Fields] OR "children"[All Fields] OR "child\'s"[All Fields] OR "children\'s"[All Fields] OR "childrens"[All Fields] OR "childs"[All Fields]'}, {'From': 'born', 'To': '"parturition"[MeSH Terms] OR "parturition"[All Fields] OR "born"[All Fields]'}, {'From': 'prematurely', 'To': '"premature birth"[MeSH Terms] OR ("premature"[All Fields] AND "birth"[All Fields]) OR "premature birth"[All Fields] OR "premature"[All Fields] OR "prematurely"[A

In [22]:
# Download details from the retrieved IDs
handle = Entrez.efetch(db="pubmed", id=record["IdList"], rettype="medline", retmode="text")
records = list(Medline.parse(handle))
handle.close()

records

[{'PMID': '37775921',
  'OWN': 'NLM',
  'STAT': 'MEDLINE',
  'DCOM': '20240112',
  'LR': '20240815',
  'IS': '1651-2227 (Electronic) 0803-5253 (Linking)',
  'VI': '113',
  'IP': '2',
  'DP': '2024 Feb',
  'TI': 'Colour perception develops throughout childhood with increased risk of deficiencies in children born prematurely.',
  'PG': '259-266',
  'LID': '10.1111/apa.16978 [doi]',
  'AB': 'AIM: To quantify the impact of prematurity on chromatic discrimination throughout childhood, from 2 to 15 years of age. METHODS: We recruited two cohorts of children, as part of the TrackAI Project, an international project with seven different study sites: a control group of full-term children with normal visual development and a group of children born prematurely. All children underwent a complete ophthalmological exam and an assessment of colour discrimination along the three colour axes: deutan, protan and trytan using a DIVE device with eye tracking technology. RESULTS: We enrolled a total of 187

In [25]:
# Extract relevant fields
final_fields = ["PMID", "TI", "AB", "AU", "OT", "MH", "JT", "DP", "LID"]
row = {
    "title": records[0].get("TI", ""),
    "abstract": records[0].get("AB", ""),
    "mesh": records[0].get("MH", ""),
    "authors": "; ".join(records[0].get("AU", [])),
    "author_keywords": records[0].get("OT", ""),
    "journal": records[0].get("JT", ""),
    "year": records[0].get("DP", "")[:4],
    "pmid": records[0].get("PMID", ""),
    "doi": records[0].get("LID", "").split()[0] if "LID" in records[0] else "",
}

row

{'title': 'Colour perception develops throughout childhood with increased risk of deficiencies in children born prematurely.',
 'abstract': 'AIM: To quantify the impact of prematurity on chromatic discrimination throughout childhood, from 2 to 15 years of age. METHODS: We recruited two cohorts of children, as part of the TrackAI Project, an international project with seven different study sites: a control group of full-term children with normal visual development and a group of children born prematurely. All children underwent a complete ophthalmological exam and an assessment of colour discrimination along the three colour axes: deutan, protan and trytan using a DIVE device with eye tracking technology. RESULTS: We enrolled a total of 1872 children (928 females and 944 males) with a mean age of 6.64 years. Out of them, 374 were children born prematurely and 1498 were full-term controls. Using data from all the children born at term, reference normative curves were plotted for colour d

## 3. Building the Ingestion Function
Wrap the search and fetch logic into a reusable function that parses the text output, filters out records without abstracts, and maps the proprietary MEDLINE keys into a clean dictionary.

In [26]:
def fetch_pubmed_abstracts(
    query: str, max_results: int = 15
) -> List[Dict[str, str]]:
    """Search PubMed and return abstracts with metadata."""
    # Fetch IDs
    handle = Entrez.esearch(db="pubmed", term=query, retmax=max_results, sort="relevance")
    record = Entrez.read(handle)
    handle.close()
    ids = record["IdList"]
    print(f"Found {len(ids)} papers for query: '{query}'")

    if not ids:
        return []

    # Download details
    handle = Entrez.efetch(db="pubmed", id=ids, rettype="medline", retmode="text")
    records = list(Medline.parse(handle))
    handle.close()

    papers = []
    for r in records:
        abstract = r.get("AB", "")
        if abstract:  # Only include papers with an abstract
            papers.append({
                "title": r.get("TI", ""),
                "abstract": abstract,
                "mesh": r.get("MH", ""),
                "authors": "; ".join(r.get("AU", [])),
                "author_keywords": r.get("OT", ""),
                "journal": r.get("JT", ""),
                "year": r.get("DP", "")[:4],
                "pmid": r.get("PMID", ""),
                "doi": r.get("LID", "").split()[0] if "LID" in r else "",
            })

    print(f"Downloaded {len(papers)} papers with abstracts")
    return papers

In [27]:
# Example usage

# Download a sample set of papers
papers = fetch_pubmed_abstracts("Colour perception deficiencies in children born prematurely", max_results=15)

# Quick preview
for p in papers[:3]:
    print(f"\n[{p['pmid']}]-[{p['doi']}] {p['title'][:80]}...")
    print(f"   {p['abstract'][:150]}...")

Found 2 papers for query: 'Colour perception deficiencies in children born prematurely'
Downloaded 2 papers with abstracts

[37775921]-[10.1111/apa.16978] Colour perception develops throughout childhood with increased risk of deficienc...
   AIM: To quantify the impact of prematurity on chromatic discrimination throughout childhood, from 2 to 15 years of age. METHODS: We recruited two coho...

[23841870]-[10.1111/dmcn.12206] Compromised approximate number system acuity in extremely preterm school-aged ch...
   AIM: The aim of this study was to compare the approximate number system acuity in children born extremely preterm aged 6 years 6 months and typically ...


## 4. Exporting Data
    
The final and most important step for this notebook is to **export the scraped data to disk**. 

The goal is to avoid pinging the NCBI API repeatedly in downstream pipelines. The embeddings generated in the next phase (`03_embedding_benchmark.ipynb`) will load this local JSON file instead of making live web requests.

In [29]:
import json
import os

# Ensure the output directory exists
output_dir = "../data/raw"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "pubmed_sample.json")

# Save the dataset to disk
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(papers, f, indent=2, ensure_ascii=False)

print(f"Successfully saved {len(papers)} papers to {output_path}")

Successfully saved 2 papers to ../data/raw/pubmed_sample.json


## 5. Bulk Ingestion (Training Corpus)

To train our model and populate the RAG vector database, we need a broad corpus of medical literature. 
Instead of querying PubMed for every single MTSamples record (~5,000 queries), we will extract the unique **40 medical specialties** and fetch 50-100 high-relevance papers for each. This builds a robust, clustered knowledge base of ~4,000 abstracts.

In [30]:
import pandas as pd
import time

# 1. Load specialties from MTSamples
MTSAMPLES_PATH = "hf://datasets/harishnair04/mtsamples/mtsamples.csv"
df = pd.read_csv(MTSAMPLES_PATH)

# Clean and extract unique specialties
specialties = df["medical_specialty"].dropna().unique()
specialties = [s.strip() for s in specialties if str(s).strip()]
print(f"Found {len(specialties)} unique specialties.\n")

# 2. Fetch papers for each specialty
all_specialty_papers = {}
papers_per_specialty = 50

# NOTE: Sliced to [:3] for safe testing. Remove the slice to run the full bulk download
for spec in specialties[:3]:
    print(f"=== Fetching for: {spec} ===")

    # You can enrich the query if needed, e.g., f"{spec} medical guidelines"
    query = spec

    papers = fetch_pubmed_abstracts(query=query, max_results=papers_per_specialty)
    all_specialty_papers[spec] = papers

    # Be respectful to the NCBI API limits (max 3 req/sec without key, 10 with key)
    time.sleep(1)

# 3. Export the bulk dataset
corpus_path = os.path.join(output_dir, "pubmed_bulk_corpus.json")
with open(corpus_path, "w", encoding="utf-8") as f:
    json.dump(all_specialty_papers, f, indent=2, ensure_ascii=False)

total_papers = sum(len(p) for p in all_specialty_papers.values())
print(f"\nBulk ingestion complete. Saved {total_papers} papers across {len(all_specialty_papers)} specialties to {corpus_path}")

Found 40 unique specialties.

=== Fetching for: Allergy / Immunology ===
Found 50 papers for query: 'Allergy / Immunology'
Downloaded 20 papers with abstracts
=== Fetching for: Bariatrics ===
Found 50 papers for query: 'Bariatrics'
Downloaded 28 papers with abstracts
=== Fetching for: Cardiovascular / Pulmonary ===
Found 50 papers for query: 'Cardiovascular / Pulmonary'
Downloaded 35 papers with abstracts

Bulk ingestion complete. Saved 83 papers across 3 specialties to ../data/raw/pubmed_bulk_corpus.json
